# **SHL Grammar Scoring Engine**

## **1. Problem Statement**

The objective of this project is to predict the grammatical quality of spoken English from audio recordings. Each recording is associated with a human-provided Mean Opinion Score (MOS) grammar label ranging from 0 to 5.

## **2. Proposed Approach**

This project uses a multimodal regression approach that combines:
- Speech transcripts generated using OpenAI Whisper.
- Word-level and character-level TF-IDF features extracted from transcripts.
- Audio and transcription diagnostic features, including duration, RMS energy, peak amplitude, Whisper average log probability, and no-speech probability.
- Ridge regression to predict the grammar score.

## **3. Evaluation**

Model performance is assessed using Root Mean Squared Error (RMSE) and Pearson correlation. Five-fold cross-validation is used to assess performance across different data splits.

## **4. Reproducibility**

The notebook reuses previously generated training transcripts and audio features. It includes data validation, preprocessing, model evaluation, final prediction generation, and submission-file checks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

# Project paths
DATA_DIR = Path.cwd()

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_CSV = DATA_DIR / "sample_submission.csv"

TRANSCRIPTS_CSV = DATA_DIR / "train_transcripts.csv"
FEATURES_CSV = DATA_DIR / "signal_features.csv"

# Load datasets
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_df = pd.read_csv(SAMPLE_CSV)
transcripts_df = pd.read_csv(TRANSCRIPTS_CSV)
features_df = pd.read_csv(FEATURES_CSV)

print("scikit-learn version:", sklearn.__version__)
print("Training records:", len(train_df))
print("Test records:", len(test_df))
print("Sample submission rows:", len(sample_df))
print("Training transcripts:", len(transcripts_df))
print("Audio feature rows:", len(features_df))

print("\nTraining columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Transcript columns:", transcripts_df.columns.tolist())
print("Feature columns:", features_df.columns.tolist())

assert {"filename", "label"}.issubset(train_df.columns)
assert {"filename", "label"}.issubset(test_df.columns)
assert train_df["filename"].is_unique
assert test_df["filename"].is_unique
assert train_df["label"].between(0, 5).all()
assert test_df["label"].eq(-1).all()

print("\nBasic dataset checks passed.")

scikit-learn version: 1.7.2
Training records: 769
Test records: 216
Sample submission rows: 204
Training transcripts: 769
Audio feature rows: 769

Training columns: ['filename', 'label']
Test columns: ['filename', 'label']
Transcript columns: ['filename', 'label', 'transcript', 'avg_logprob', 'avg_no_speech_prob', 'status']
Feature columns: ['filename', 'label', 'duration_sec', 'rms', 'peak', 'near_silence_fraction']

Basic dataset checks passed.


In [2]:
# Verify the data needed for the multimodal model
assert transcripts_df["filename"].is_unique
assert features_df["filename"].is_unique

assert set(train_df["filename"]) == set(transcripts_df["filename"])
assert set(train_df["filename"]) == set(features_df["filename"])

print("All training filenames match across the three datasets.")
print("\nTranscript status counts:")
print(transcripts_df["status"].value_counts(dropna=False))

print("\nExample transcript:")
print(transcripts_df[["filename", "label", "transcript"]].head(3))

print("\nExample audio features:")
print(features_df.head(3))

All training filenames match across the three datasets.

Transcript status counts:
status
ok    769
Name: count, dtype: int64

Example transcript:
        filename  label                                         transcript
0  audio_192.wav    3.0  The plate land is very, very beautiful and fun...
1  audio_436.wav    3.0  The scene of a school playground, our playgrou...
2  audio_447.wav    3.5  My favorite place to reject is the beach. I lo...

Example audio features:
        filename  label  duration_sec       rms      peak  \
0  audio_192.wav    3.0     60.508375  0.049853  0.477783   
1  audio_436.wav    3.0     60.074688  0.088254  0.945648   
2  audio_447.wav    3.5     60.074688  0.074962  0.919312   

   near_silence_fraction  
0               0.427317  
1               0.090846  
2               0.152531  


## **5. Feature Preparation**

The transcript and audio-feature datasets are merged using the recording filename. Word-level and character-level TF-IDF features capture lexical and grammatical patterns, while numerical features provide additional information about audio quality and transcription confidence.

All preprocessing is fitted within the model pipeline to prevent information leakage between training and validation folds.

In [3]:

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Merge by filename, never by row position
model_df = transcripts_df.merge(
    features_df.drop(columns=["label"], errors="ignore"),
    on="filename",
    how="inner",
    validate="one_to_one"
)

# Ensure every training record has exactly one merged row
assert len(model_df) == len(train_df) == 769
assert model_df["filename"].is_unique

# Use the labels from the original training CSV
model_df = model_df.drop(columns=["label"], errors="ignore").merge(
    train_df[["filename", "label"]],
    on="filename",
    how="inner",
    validate="one_to_one"
)

text_col = "transcript"

numeric_cols = [
    "avg_logprob",
    "avg_no_speech_prob",
    "duration_sec",
    "rms",
    "peak",
    "near_silence_fraction",
]

assert model_df[numeric_cols].notna().all().all()

# Word and character TF-IDF features
preprocessor = ColumnTransformer(
    transformers=[
        (
            "word_tfidf",
            TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                max_features=20000,
                sublinear_tf=True
            ),
            text_col
        ),
        (
            "char_tfidf",
            TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
                max_features=30000,
                sublinear_tf=True
            ),
            text_col
        ),
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_cols
        )
    ],
    remainder="drop"
)

X = model_df[[text_col] + numeric_cols]
y = model_df["label"].astype(float)

print("Training matrix rows:", len(X))
print("Numeric features:", numeric_cols)
print("Target score range:", y.min(), "to", y.max())
print("Mean target score:", round(y.mean(), 4))

Training matrix rows: 769
Numeric features: ['avg_logprob', 'avg_no_speech_prob', 'duration_sec', 'rms', 'peak', 'near_silence_fraction']
Target score range: 0.0 to 5.0
Mean target score: 3.3134


## **6. Model Training and Evaluation**

Ridge regression is used as the baseline regression model. The regularization parameter is set to alpha = 1.0 based on the previous tuning experiment.

Five-fold cross-validation is used to estimate generalization performance. RMSE measures prediction error, while Pearson correlation measures the strength of the linear relationship between predicted and actual scores.

In [4]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, root_mean_squared_error
from scipy.stats import pearsonr

def pearson_correlation(y_true, y_pred):
    if np.std(y_pred) == 0 or np.std(y_true) == 0:
        return 0.0
    return pearsonr(y_true, y_pred)[0]

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "rmse": make_scorer(
        root_mean_squared_error,
        greater_is_better=False
    ),
    "pearson": make_scorer(pearson_correlation)
}

cv_results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1
)

train_rmse = -cv_results["train_rmse"]
val_rmse = -cv_results["test_rmse"]
val_pearson = cv_results["test_pearson"]

print("Training RMSE per fold:", np.round(train_rmse, 4))
print("Validation RMSE per fold:", np.round(val_rmse, 4))
print("Pearson per fold:", np.round(val_pearson, 4))
print(f"\nMean training RMSE: {train_rmse.mean():.4f}")
print(f"Mean validation RMSE: {val_rmse.mean():.4f}")
print(f"Validation RMSE std: {val_rmse.std():.4f}")
print(f"Mean Pearson correlation: {val_pearson.mean():.4f}")

Training RMSE per fold: [0.3086 0.3262 0.3197 0.3237 0.3215]
Validation RMSE per fold: [0.9004 0.8162 0.8596 0.818  0.8316]
Pearson per fold: [0.7547 0.749  0.6805 0.7712 0.7141]

Mean training RMSE: 0.3199
Mean validation RMSE: 0.8452
Validation RMSE std: 0.0317
Mean Pearson correlation: 0.7339


## **7. Final Model Training**

After evaluating the model using five-fold cross-validation, the selected Ridge regression pipeline is retrained using all 769 labeled training recordings. This final model is then used to predict grammar scores for the unlabeled test recordings.

In [5]:
# Train the selected model on the full labeled dataset.
# The CV results above remain the reported validation metrics.

final_model = model.fit(X, y)

print("Final model trained on:", len(X), "recordings")
print("Selected model: Ridge Regression")
print("Alpha:", final_model.named_steps["regressor"].alpha)
print("Training data mean score:", round(y.mean(), 4))

Final model trained on: 769 recordings
Selected model: Ridge Regression
Alpha: 1.0
Training data mean score: 3.3134


In [6]:
print("Test folder:", DATA_DIR / "test")
print("Test audio files:", len(list((DATA_DIR / "test").glob("*.wav"))))

print("\nCSV files in the project folder:")
for path in sorted(DATA_DIR.glob("*.csv")):
    print(path.name)

Test folder: d:\AI Projects\shl-hiring-assessment-2026\Dataset_Final\test
Test audio files: 216

CSV files in the project folder:
audio_metadata.csv
cross_validation_results.csv
ridge_tuning_results.csv
sample_submission.csv
sample_transcripts.csv
signal_features.csv
submission_candidate.csv
test.csv
test_predictions.csv
test_signal_features.csv
test_transcripts.csv
train.csv
train_transcripts.csv
validation_multimodal_predictions.csv
validation_predictions.csv


## **8. Test Prediction Generation**

The fitted pipeline is applied to the 216 test recordings. Test transcripts and signal features must have been generated beforehand using the same extraction process as the training data. Predictions are clipped to the valid 0–5 score range.

In [7]:
# Generate test predictions from the fitted multimodal model
TEST_TRANSCRIPTS_CSV = DATA_DIR / "test_transcripts.csv"
TEST_FEATURES_CSV = DATA_DIR / "test_signal_features.csv"

test_transcripts_df = pd.read_csv(TEST_TRANSCRIPTS_CSV)
test_features_df = pd.read_csv(TEST_FEATURES_CSV)

assert test_transcripts_df["filename"].is_unique
assert test_features_df["filename"].is_unique
assert set(test_df["filename"]) == set(test_transcripts_df["filename"])
assert set(test_df["filename"]) == set(test_features_df["filename"])
assert test_transcripts_df["status"].eq("ok").all()

test_model_df = test_transcripts_df.merge(
    test_features_df.drop(columns=["label"], errors="ignore"),
    on="filename",
    how="inner",
    validate="one_to_one"
)

# Keep the feature names and order identical to training
X_test = test_model_df[[text_col] + numeric_cols].copy()
assert list(X_test.columns) == list(X.columns)
assert len(X_test) == len(test_df) == 216
assert X_test["transcript"].notna().all()

raw_predictions = final_model.predict(X_test)
test_predictions = np.clip(raw_predictions, 0, 5)

prediction_df = pd.DataFrame({
    "filename": test_model_df["filename"],
    "label": test_predictions
})

# Restore the exact filename order in test.csv
prediction_df = test_df[["filename"]].merge(
    prediction_df,
    on="filename",
    how="left",
    validate="one_to_one"
)

assert len(prediction_df) == 216
assert prediction_df["label"].notna().all()
assert np.isfinite(prediction_df["label"]).all()
assert prediction_df["label"].between(0, 5).all()
assert prediction_df["filename"].is_unique

prediction_path = DATA_DIR / "test_predictions.csv"
prediction_df.to_csv(prediction_path, index=False)

print("Predictions generated:", len(prediction_df))
print("Missing predictions:", prediction_df["label"].isna().sum())
print("Duplicate filenames:", prediction_df["filename"].duplicated().sum())
print("Prediction range:", round(prediction_df["label"].min(), 4), "to", round(prediction_df["label"].max(), 4))
print("Mean prediction:", round(prediction_df["label"].mean(), 4))
print("Saved:", prediction_path)
display(prediction_df.head(10))

Predictions generated: 216
Missing predictions: 0
Duplicate filenames: 0
Prediction range: 0.0 to 5.0
Mean prediction: 3.2665
Saved: d:\AI Projects\shl-hiring-assessment-2026\Dataset_Final\test_predictions.csv


,filename,label
0,audio_128.wav,2.553292
1,audio_156.wav,4.048230
2,audio_19.wav,4.289685
3,audio_108.wav,3.141378
4,audio_206.wav,2.719719
5,audio_161.wav,2.541165
6,audio_215.wav,2.296373
7,audio_213.wav,1.808694
8,audio_11.wav,3.661539
9,audio_155.wav,3.156788


## **9. Submission Candidate and Final Checks**

The sample submission currently has 204 rows, while the test CSV has 216. This notebook creates a 216-row candidate using the test CSV filenames in their original order; confirm the competition's official instructions/schema before uploading.

In [8]:
# Create a candidate submission without modifying the supplied sample file
submission = test_df[["filename"]].merge(
    prediction_df[["filename", "label"]],
    on="filename",
    how="left",
    validate="one_to_one"
)

assert len(submission) == len(test_df) == 216
assert submission["filename"].equals(test_df["filename"])
assert submission["label"].notna().all()
assert np.isfinite(submission["label"]).all()
assert submission["label"].between(0, 5).all()

submission_path = DATA_DIR / "submission_candidate.csv"
submission.to_csv(submission_path, index=False)

print("Submission candidate:", submission_path)
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Missing scores:", submission["label"].isna().sum())
print("Duplicate filenames:", submission["filename"].duplicated().sum())
print("Sample submission rows:", len(sample_df))
print("Test rows:", len(test_df))
print("Note: confirm the official competition submission requirements before uploading.")
display(submission.head(10))

Submission candidate: d:\AI Projects\shl-hiring-assessment-2026\Dataset_Final\submission_candidate.csv
Rows: 216
Columns: ['filename', 'label']
Missing scores: 0
Duplicate filenames: 0
Sample submission rows: 204
Test rows: 216
Note: confirm the official competition submission requirements before uploading.


,filename,label
0,audio_128.wav,2.553292
1,audio_156.wav,4.048230
2,audio_19.wav,4.289685
3,audio_108.wav,3.141378
4,audio_206.wav,2.719719
5,audio_161.wav,2.541165
6,audio_215.wav,2.296373
7,audio_213.wav,1.808694
8,audio_11.wav,3.661539
9,audio_155.wav,3.156788


# **Project Author and Contributions**

**Author:** Ittyavira C. Abraham  
**Project:** SHL Hiring Assessment 2026 — Grammar Scoring Engine  
**Objective:** Predict a continuous spoken-English grammar score between 0 and 5 from audio recordings.

## **Work completed**

- Audited the training and test CSV files and checked filename consistency, missing values, and duplicate identifiers.
- Transcribed the training and test audio recordings with OpenAI Whisper and checked transcription status and coverage.
- Extracted basic acoustic features from each recording: duration, root-mean-square (RMS) amplitude, peak amplitude, and near-silence fraction.
- Combined transcript text features (word-level and character-level TF-IDF) with numerical audio and transcription metadata features.
- Built a scikit-learn Ridge Regression pipeline with numerical preprocessing.
- Evaluated the model using shuffled five-fold cross-validation and reported validation RMSE and Pearson correlation.
- Retrained the selected pipeline on the full labeled training set and generated predictions for the test recordings.
- Clipped predictions to the valid score range of 0–5 and validated the output CSV for row count, missing predictions, duplicate filenames, and score range.

## **Model evaluation**

The notebook's five-fold cross-validation results are the reported validation results. The mean validation RMSE is approximately **0.8452** and the mean fold-wise Pearson correlation is approximately **0.7339**. These are local validation metrics, not a Kaggle leaderboard score.

## **Deliverables**

- `SHL_Grammar_Scoring_Engine_Final.ipynb` — documented workflow, model evaluation, and prediction code.
- `test_predictions.csv` — model predictions for the 216 rows listed in `test.csv`.
- `submission_candidate.csv` — validated 216-row submission candidate with `filename` and `label` columns.

## **Notes and limitations**

The provided `sample_submission.csv` contains 204 rows, whereas `test.csv` contains 216 rows. The candidate preserves all 216 test filenames in `test.csv` order. The final upload format should be confirmed against the competition's current instructions. The test labels are unavailable locally, so the true test RMSE cannot be calculated from this dataset.